# Academic calendar and public safety

**A3.** Do daily noise-call and recorded assault-case counts change more between semester and winter break near campus than in other Berkeley beats?

**A4.** How do noise, assault and alcohol-related collision counts on event dates compare with matched ordinary dates?

Both analyses use `data/calendar/daily_calendar.csv`. A4 reads and writes these files:

| Comparison | Input | Output |
| --- | --- | --- |
| Alcohol-related collisions | `data/processed/collisions.csv` | `data/processed/a4_collision_matches.csv` |
| Noise calls | `data/processed/calls_2021_2025.csv.gz` | `data/processed/a4_noise_matches.csv` |
| Recorded assaults | `data/processed/crime_cases_2021_2025.csv.gz` | `data/processed/a4_assault_matches.csv` |

A3 uses the shared address lookup and beat polygons. Its tables are `data/processed/a3_period_rates.csv` and `data/processed/a3_unassigned.csv`.

The event-day report table is saved in `results/04_event_summary.csv`.

## A3 · Semester and winter break near campus

Compare daily noise-call counts and recorded assault cases in Southside (sectors 6 and 7) with the remaining matched beats. Repeat with sectors 1, 4, 6, and 7. Campus itself is outside the beat map. Report battery calls separately as a comparison series.

For each outcome, use one row per event, the source's verified reporting dates, and the shared sector definitions with `bpdtools.calendar_rates.period_rates`. For assaults, filter the shared one-row-per-case table using `ASSAULT_INCIDENT_TYPES` and its local occurrence date. For calls, use the shared codebook's event categories and sector assignments. Report unassigned records by period. Days with coverage and zero events enter the denominator. Battery calls include officer-initiated records. All three series count recorded events per covered day.

In [1]:
from pathlib import Path

import pandas as pd

# Find the data directory when running from the repository root or notebooks folder.
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
inputs = root / "data"
# Require one row per local date so calendar lookups identify a single day.
calendar = pd.read_csv(inputs / "calendar/daily_calendar.csv", parse_dates=["date"])
if not calendar["date"].is_unique:
    raise ValueError("Expected one calendar row per date")
calendar = calendar.set_index("date").sort_index()

In [2]:
from bpdtools.calendar_rates import period_rates
from bpdtools.calls import covered_days
from bpdtools.config import ASSAULT_INCIDENT_TYPES, BATTERY_CALL_TYPES, NOISE_CALL_TYPES
from bpdtools.sectors import (
    assign_case_sectors,
    attach_address_sectors,
    load_sector_features,
)

processed = inputs / "processed"
lookup = pd.read_csv(processed / "address_sectors.csv", dtype={"block_key": str})
calls = pd.read_csv(processed / "calls_2021_2025.csv.gz", parse_dates=["date"])
cases = pd.read_csv(processed / "crime_cases_2021_2025.csv.gz", parse_dates=["date"])
call_days = covered_days(calls["date"])
case_days = covered_days(cases["date"])
# Assign geography only to records in the study calendar.
calls = calls.loc[calls.date.isin(calendar.index)]
cases = cases.loc[
    cases.date.isin(calendar.index) & cases.Incident_Type.isin(ASSAULT_INCIDENT_TYPES)
]
calls = attach_address_sectors(calls, lookup).set_index("Incident Number")
cases = assign_case_sectors(cases, load_sector_features(), lookup).set_index(
    "Case_Number"
)
outcomes = {
    "noise": (calls.loc[calls["Call Type"].isin(NOISE_CALL_TYPES)], call_days),
    "assault": (cases, case_days),
    "battery_calls": (
        calls.loc[calls["Call Type"].isin(BATTERY_CALL_TYPES)],
        call_days,
    ),
}
rate_tables = {}
audit_tables = {}
for outcome, (events_in_calendar, days) in outcomes.items():
    for definition, sectors in {"southside": [6, 7], "adjacent": [1, 4, 6, 7]}.items():
        rates, audit = period_rates(
            events_in_calendar, calendar.reset_index(), days, sectors
        )
        rate_tables[outcome, definition] = rates
        audit_tables[outcome] = (
            audit  # Missing geography is independent of the near definition.
        )
rates = pd.concat(rate_tables, names=["outcome", "definition"])
audit = pd.concat(audit_tables, names=["outcome"])
rates.to_csv(processed / "a3_period_rates.csv")
audit.to_csv(processed / "a3_unassigned.csv")
display(rates.round(2), audit.round(3))

events  covered_days  daily_rate  \
outcome       definition area  period                                           
noise         southside  near  semester         455           122        3.73   
                               winter_break      17            24        0.71   
                         other semester         527           122        4.32   
                               winter_break      55            24        2.29   
              adjacent   near  semester         580           122        4.75   
                               winter_break      26            24        1.08   
                         other semester         402           122        3.30   
                               winter_break      46            24        1.92   
assault       southside  near  semester          52           122        0.43   
                               winter_break       8            24        0.33   
                         other semester         199           122        1.63   
                               winter_break      36            24        1.50   
              adjacent   near  semester          97           122        0.80   
                               winter_break      19            24        0.79   
                         other semester         154           122        1.26   
                               winter_break      25            24        1.04   
battery_calls southside  near  semester          81           122        0.66   
                               winter_break      12            24        0.50   
                         other semester         300           122        2.46   
                               winter_break      49            24        2.04   
              adjacent   near  semester         142           122        1.16   
                               winter_break      23            24        0.96   
                         other semester         239           122        1.96   
                               winter_break      38            24        1.58   

                                             semester_to_break_ratio  
outcome       definition area  period                                 
noise         southside  near  semester                         5.27  
                               winter_break                     5.27  
                         other semester                         1.88  
                               winter_break                     1.88  
              adjacent   near  semester                         4.39  
                               winter_break                     4.39  
                         other semester                         1.72  
                               winter_break                     1.72  
assault       southside  near  semester                         1.28  
                               winter_break                     1.28  
                         other semester                         1.09  
                               winter_break                     1.09  
              adjacent   near  semester                         1.00  
                               winter_break                     1.00  
                         other semester                         1.21  
                               winter_break                     1.21  
battery_calls southside  near  semester                         1.33  
                               winter_break                     1.33  
                         other semester                         1.20  
                               winter_break                     1.20  
              adjacent   near  semester                         1.21  
                               winter_break                     1.21  
                         other semester                         1.24  
                               winter_break                     1.24

total  unassigned  unassigned_share
outcome       period                                           
noise         semester        995          13             0.013
              winter_break     72           0             0.000
assault       semester        261          10             0.038
              winter_break     44           0             0.000
battery_calls semester        383           2             0.005
              winter_break     61           0             0.000

### What changed during the semester?

Across 122 semester days and 24 winter-break days, Southside averaged 3.73 noise calls per day during the semester and 0.71 during break (5.27 times as many). Other mapped beats had a smaller ratio of 1.88. Expanding the near-campus group to sectors 1, 4, 6 and 7 preserved this pattern (4.39 versus 1.72).

Recorded assaults showed a smaller and geographically sensitive contrast. The ratios were 1.28 in Southside and 1.09 elsewhere, but 1.00 and 1.21 under the broader campus definition. Battery-call ratios were also close under the broader definition (1.21 versus 1.24). The semester had 13 unassigned noise calls, 10 unassigned assault cases and two unassigned battery calls; all break records in these categories had a sector. These counts measure BPD reports during one semester and its winter break. They do not adjust for changes in the number of people present.

## A4 · Fall 2024 event-day comparisons

How many alcohol-related collisions did BPD record on home football, new-student move-in (August 20–21), Halloween, and finals dates compared with ordinary days? We count a collision when its detailed primary collision factor explicitly mentions alcohol or a blood alcohol concentration threshold (`alcohol_pcf_status == "explicit_alcohol"`).

For each event day, average the counts on eligible dates in the same year and month that fall on the same weekday. A control date can serve multiple events.

**Inputs**  
`data/calendar/daily_calendar.csv` and `data/processed/collisions.csv`

**Output**  
`data/processed/a4_collision_matches.csv`

In [3]:
# Use the local collision date to match the Berkeley calendar.
collisions = pd.read_csv(
    inputs / "processed/collisions.csv", usecols=["Date", "alcohol_pcf_status"]
)
collisions["Date"] = pd.to_datetime(collisions["Date"], format="%Y-%m-%d")
print(f"Snapshot records without a local date: {collisions['Date'].isna().sum()}")

Snapshot records without a local date: 101


In [4]:
events = ["home_football", "move_in", "halloween", "finals"]
excluded = events + [
    "holiday",
    "non_instructional",
    "rrr",
    "instruction_begins",
    "formal_classes_end",
    "last_instruction_day",
    "winter_break",
]
# A semester day can serve as a control if all named event and exclusion flags are zero.
ordinary = calendar.index[calendar[excluded].eq(0).all(axis=1)]

# Zero here means zero dated records flagged in this snapshot on that day.
flagged = collisions.loc[collisions["alcohol_pcf_status"].eq("explicit_alcohol")]
counts = flagged.groupby("Date").size().reindex(calendar.index, fill_value=0)

# Preserve unmatched event dates with an empty control pool and missing control mean.
records = []
for event in events:
    for day in calendar.index[calendar[event].eq(1)]:
        # Match the weekday and calendar month to account for seasonality.
        controls = ordinary[
            (ordinary.to_period("M") == day.to_period("M"))
            & (ordinary.day_of_week == day.weekday())
        ]
        records.append(
            {
                "event": event,
                "event_date": day.date().isoformat(),
                "control_dates": ", ".join(controls.strftime("%Y-%m-%d")),
                "n_controls": len(controls),
                "event_alcohol_collisions": counts.at[day],
                "mean_control_alcohol_collisions": counts.loc[controls].mean(),
            }
        )

matches = pd.DataFrame(records)
output = root / "data/processed/a4_collision_matches.csv"
matches.to_csv(output, index=False)
matches

,event,event_date,control_dates,n_controls,event_alcohol_collisions,mean_control_alcohol_collisions
0,home_football,2024-08-31,2024-08-24,1,0,0.000000
1,home_football,2024-09-14,"2024-09-07, 2024-09-21, 2024-09-28",3,0,0.333333
2,home_football,2024-10-05,2024-10-12,1,0,0.000000
3,home_football,2024-10-19,2024-10-12,1,0,0.000000
4,home_football,2024-10-26,2024-10-12,1,0,0.000000
5,home_football,2024-11-16,"2024-11-02, 2024-11-09, 2024-11-30",3,1,0.000000
6,home_football,2024-11-23,"2024-11-02, 2024-11-09, 2024-11-30",3,1,0.000000
7,move_in,2024-08-20,2024-08-27,1,0,0.000000
8,move_in,2024-08-21,,0,0,NaN
9,halloween,2024-10-31,"2024-10-03, 2024-10-10, 2024-10-17, 2024-10-24",4,0,0.000000


In [5]:
# Each matched event date receives equal weight, regardless of its pool size.
matched = matches.loc[matches["n_controls"].gt(0)]
summary = matches.groupby("event").agg(
    event_days=("event_date", "size"),
    all_event_alcohol_collisions=("event_alcohol_collisions", "sum"),
)
summary.join(
    matched.groupby("event").agg(
        matched_days=("event_date", "size"),
        mean_event_alcohol_collisions=("event_alcohol_collisions", "mean"),
        mean_control_alcohol_collisions=("mean_control_alcohol_collisions", "mean"),
    )
)

,event_days,all_event_alcohol_collisions,matched_days,mean_event_alcohol_collisions,mean_control_alcohol_collisions
event,,,,,
finals,5,0,4,0.000000,0.000000
halloween,1,0,1,0.000000,0.000000
home_football,7,2,7,0.285714,0.047619
move_in,2,0,1,0.000000,0.000000


### Interpretation

All seven home games had eligible comparisons. Two flagged collisions occurred on game days. Move-in on August 21 and finals on December 20 had no eligible comparison date; their control means are missing. The sparse counts describe recorded primary factors and support only a descriptive comparison for this semester. The noise and assault comparisons follow below.

### A4 · Citywide noise calls

Did BPD receive more noise calls on event dates than on their matched ordinary dates? We use the same dates as the collision comparison above. Each incident in the shared noise call categories counts once. We establish coverage from calls of all types before assigning a zero to a day.

**Input:** `data/processed/calls_2021_2025.csv.gz` and the calendar matches above.  
**Output:** `data/processed/a4_noise_matches.csv`, one row per event date.

In [6]:
from bpdtools.calls import covered_days
from bpdtools.config import NOISE_CALL_TYPES

calls = pd.read_csv(
    inputs / "processed/calls_2021_2025.csv.gz",
    usecols=["date", "Call Type"],
    parse_dates=["date"],
)
# Check the full calendar against all calls before counting just noise calls.
reporting_days = covered_days(calls["date"])
if not calendar.index.isin(reporting_days).all():
    raise ValueError("Check CFS coverage before using these calendar matches.")
noise_calls = calls.loc[calls["Call Type"].isin(NOISE_CALL_TYPES)]
daily_noise = noise_calls.groupby("date").size().reindex(calendar.index, fill_value=0)


def match_daily_counts(daily_counts, outcome):
    """Add event-day counts and control-day means to the calendar matches."""
    result = matches[["event", "event_date", "control_dates", "n_controls"]].copy()
    result[f"event_{outcome}"] = pd.to_datetime(result["event_date"]).map(daily_counts)
    # An empty pool has an undefined mean. A covered day can have a true zero.
    control_means = []
    for pool in result["control_dates"]:
        days = pd.to_datetime(pool.split(", ")) if pool else pd.DatetimeIndex([])
        control_means.append(daily_counts.loc[days].mean())
    result[f"mean_control_{outcome}"] = control_means
    return result


noise_matches = match_daily_counts(daily_noise, "noise_calls")
noise_matches.to_csv(inputs / "processed/a4_noise_matches.csv", index=False)

# Compare only matched event days and give each event day equal weight.
noise_matches.loc[noise_matches["n_controls"].gt(0)].groupby("event").agg(
    matched_days=("event_date", "size"),
    mean_event_noise_calls=("event_noise_calls", "mean"),
    mean_control_noise_calls=("mean_control_noise_calls", "mean"),
)

,matched_days,mean_event_noise_calls,mean_control_noise_calls
event,,,
finals,4,4.000000,4.25
halloween,1,16.000000,6.00
home_football,7,19.714286,17.00
move_in,1,7.000000,6.00


Across seven home football dates, BPD recorded an average of 19.7 noise calls per day, compared with 17.0 on their matched ordinary days. Five games had more calls than their own comparison mean. Halloween had 16 calls against a comparison mean of 6, but it contributes just one event date. During finals, the four matched dates averaged 4.0 calls against 4.25. Only the first of the two move-in dates had a match (7 calls against 6). Some football dates share controls, so these comparisons describe this semester and do not establish an event effect.

### A4 · Citywide recorded assault cases

Count each qualifying misdemeanor or felony assault/battery case once on its local occurrence date. The citywide comparison includes cases with missing coordinates. Check dates against the full crime table before filling days with zero assaults.

In [7]:
from bpdtools.config import ASSAULT_INCIDENT_TYPES

cases = pd.read_csv(
    inputs / "processed/crime_cases_2021_2025.csv.gz",
    usecols=["Case_Number", "date", "Incident_Type"],
    parse_dates=["date"],
)
if cases["Case_Number"].isna().any() or not cases["Case_Number"].is_unique:
    raise ValueError("Expected one row per recorded case.")
if not calendar.index.isin(cases["date"]).all():
    raise ValueError("Check crime data coverage before using these matches.")
assaults = cases.loc[cases["Incident_Type"].isin(ASSAULT_INCIDENT_TYPES)]
daily_assaults = assaults.groupby("date").size().reindex(calendar.index, fill_value=0)
assault_matches = match_daily_counts(daily_assaults, "assault_cases")
assault_matches.to_csv(inputs / "processed/a4_assault_matches.csv", index=False)
assault_matches.loc[assault_matches["n_controls"].gt(0)].groupby("event").agg(
    matched_days=("event_date", "size"),
    mean_event_cases=("event_assault_cases", "mean"),
    mean_control_cases=("mean_control_assault_cases", "mean"),
)

,matched_days,mean_event_cases,mean_control_cases
event,,,
finals,4,1.750000,2.250000
halloween,1,1.000000,5.000000
home_football,7,1.857143,3.619048
move_in,1,3.000000,1.000000


Across seven home football games, BPD recorded 1.86 assault cases per game day on average, compared with 3.62 on matched ordinary days. The four matched finals dates averaged 1.75 versus 2.25. Halloween had one case versus a comparison mean of five. Of the two new-student move-in dates, only August 20 had a match (three cases versus one). The small counts, single-date comparisons, and shared controls limit what we can infer about other semesters.

In [8]:
# Average over matched event dates, giving each date equal weight across outcomes.
event_rows = matches.groupby("event", sort=False)
event_summary = event_rows.size().to_frame("event_days")
event_summary["matched_days"] = event_rows["n_controls"].agg(lambda n: n.gt(0).sum())
for table, outcome in [
    (noise_matches, "noise_calls"),
    (assault_matches, "assault_cases"),
    (matches, "alcohol_collisions"),
]:
    paired = table.loc[table["n_controls"].gt(0)].groupby("event")
    for column in [f"event_{outcome}", f"mean_control_{outcome}"]:
        event_summary[column] = paired[column].mean()

event_summary.to_csv(root / "results/04_event_summary.csv")
event_summary

,event_days,matched_days,event_noise_calls,mean_control_noise_calls,event_assault_cases,mean_control_assault_cases,event_alcohol_collisions,mean_control_alcohol_collisions
event,,,,,,,,
home_football,7,7,19.714286,17.00,1.857143,3.619048,0.285714,0.047619
move_in,2,1,7.000000,6.00,3.000000,1.000000,0.000000,0.000000
halloween,1,1,16.000000,6.00,1.000000,5.000000,0.000000,0.000000
finals,5,4,4.000000,4.25,1.750000,2.250000,0.000000,0.000000
